## Imports y Configuración

Define el video, la salida y el bloque que se desea probar.


In [1]:
from pathlib import Path
import math
import shutil

import cv2
import numpy as np

VIDEO = Path("Videos") / "video30min-11to22.mp4"
SALIDA = Path("Videos")

FPS_SALIDA = 5.0
SEGUNDO_INICIO = None
SEGUNDO_FIN = None

FRAMES_PRINCIPALES_POR_BLOQUE = 500
SOLAPAMIENTO_TOTAL = 20
BLOQUE_A_PROBAR = 1

EXTRAER_FRAMES_NUEVAMENTE = False
LIMPIAR_BLOQUE_ANTERIOR = True
GUARDAR_DIAGNOSTICOS = True
COMPRESION_PNG = 3


## Extracción de Frames

Extrae todos los frames si la carpeta todavía no existe o si se solicita una extracción nueva.


In [2]:
def guardar_png(ruta, imagen):
    ruta = Path(ruta)
    ruta.parent.mkdir(parents=True, exist_ok=True)
    correcto = cv2.imwrite(
        str(ruta),
        imagen,
        [cv2.IMWRITE_PNG_COMPRESSION, COMPRESION_PNG],
    )
    if not correcto:
        raise RuntimeError(f"No se pudo guardar {ruta}")


def limpiar_png(carpeta):
    carpeta = Path(carpeta)
    if carpeta.exists():
        for archivo in carpeta.glob("*.png"):
            archivo.unlink()


def extraer_frames(
    video=VIDEO,
    salida=SALIDA,
    fps_salida=FPS_SALIDA,
    segundo_inicio=SEGUNDO_INICIO,
    segundo_fin=SEGUNDO_FIN,
    forzar=EXTRAER_FRAMES_NUEVAMENTE,
):
    video = Path(video)
    salida = Path(salida)
    base = salida / video.stem
    carpeta_frames = base / "frames"

    existentes = sorted(carpeta_frames.glob("*.png"))
    if existentes and not forzar:
        print(f"Se usarán {len(existentes)} frames existentes")
        return base

    if not video.is_file():
        raise FileNotFoundError(video)

    carpeta_frames.mkdir(parents=True, exist_ok=True)
    limpiar_png(carpeta_frames)

    captura = cv2.VideoCapture(str(video))
    fps_video = float(captura.get(cv2.CAP_PROP_FPS))
    total_video = int(captura.get(cv2.CAP_PROP_FRAME_COUNT))

    if fps_video <= 0 or total_video <= 0:
        captura.release()
        raise RuntimeError("No se pudieron leer los datos del video")

    duracion = total_video / fps_video
    inicio = 0.0 if segundo_inicio is None else max(0.0, float(segundo_inicio))
    fin = duracion if segundo_fin is None else min(duracion, float(segundo_fin))

    if fin <= inicio:
        captura.release()
        raise ValueError("El intervalo indicado no es válido")

    tiempos = np.arange(inicio, fin, 1.0 / fps_salida, dtype=np.float64)
    digitos = max(4, len(str(max(0, len(tiempos) - 1))))
    guardados = 0

    for indice, tiempo in enumerate(tiempos):
        captura.set(cv2.CAP_PROP_POS_MSEC, float(tiempo) * 1000.0)
        correcto, frame = captura.read()
        if not correcto:
            break

        guardar_png(carpeta_frames / f"{indice:0{digitos}d}.png", frame)
        guardados += 1

        if guardados % 100 == 0:
            print(f"Frames: {guardados}/{len(tiempos)}")

    captura.release()
    print(f"Frames extraídos: {guardados}")
    return base


## Plan del Bloque

Calcula los 500 frames principales y reparte el solapamiento total entre ambos lados.


In [3]:
def calcular_bloques(
    total_frames,
    frames_principales=FRAMES_PRINCIPALES_POR_BLOQUE,
    solapamiento_total=SOLAPAMIENTO_TOTAL,
):
    contexto_izquierdo = solapamiento_total // 2
    contexto_derecho = solapamiento_total - contexto_izquierdo
    cantidad = math.ceil(total_frames / frames_principales)
    bloques = []

    for numero in range(cantidad):
        inicio = numero * frames_principales
        fin = min(total_frames, inicio + frames_principales)
        inicio_contexto = max(0, inicio - contexto_izquierdo)
        fin_contexto = min(total_frames, fin + contexto_derecho)

        bloques.append({
            "numero": numero + 1,
            "inicio": inicio,
            "fin": fin,
            "inicio_contexto": inicio_contexto,
            "fin_contexto": fin_contexto,
            "principales": fin - inicio,
            "total": fin_contexto - inicio_contexto,
        })

    return bloques


def mostrar_bloques(base):
    frames = sorted((Path(base) / "frames").glob("*.png"))
    bloques = calcular_bloques(len(frames))
    print(f"Frames disponibles: {len(frames)}")
    print(f"Bloques necesarios: {len(bloques)}")

    for bloque in bloques:
        print(
            f"Bloque {bloque['numero']:02d}: "
            f"{bloque['principales']} principales, "
            f"{bloque['total']} para procesar"
        )

    return bloques


## Configuración del HUD

Prioriza la cobertura del verde fluorescente y la reconstrucción completa de las figuras.


In [4]:
VERDE_FUERTE = (37, 90, 95, 45, 13, 10)
VERDE_MEDIO = (31, 102, 32, 24, 5, 4)
VERDE_TENUE = (27, 110, 12, 12, 1, 1)

MUESTRAS_ATLAS = 300
UMBRAL_ATLAS = 0.10
RADIO_ATLAS = 7

DX_INICIAL = 0.080
DY_INICIAL = 0.082
DX_MINIMO = 0.060
DX_MAXIMO = 0.125
DY_MINIMO = 0.055
DY_MAXIMO = 0.145
PASO_BUSQUEDA = 3

GROSOR_CROSSHAIR = 10
MARGEN_CROSSHAIR = 4
RADIO_RESIDUAL_CROSSHAIR = 22

GROSOR_CIRCULO = 12
MARGEN_CIRCULO = 5
RADIO_RESIDUAL_CIRCULO = 20


## Detección del HUD

Obtiene evidencia por color únicamente dentro de las regiones donde aparece el HUD.


In [5]:
def detectar_verde(frame, configuracion):
    h_min, h_max, s_min, v_min, diferencia_rojo, diferencia_azul = configuracion
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    azul, verde, rojo = cv2.split(frame)

    azul = azul.astype(np.int16)
    verde = verde.astype(np.int16)
    rojo = rojo.astype(np.int16)

    rango = cv2.inRange(
        hsv,
        np.array([h_min, s_min, v_min], dtype=np.uint8),
        np.array([h_max, 255, 255], dtype=np.uint8),
    )
    dominio = (
        (verde >= rojo + diferencia_rojo)
        & (verde >= azul + diferencia_azul)
    ).astype(np.uint8) * 255

    return cv2.bitwise_and(rango, dominio)


def crear_zona(forma, cajas):
    alto, ancho = forma
    mascara = np.zeros(forma, dtype=np.uint8)

    for x1, y1, x2, y2 in cajas:
        cv2.rectangle(
            mascara,
            (round(ancho * x1), round(alto * y1)),
            (round(ancho * x2), round(alto * y2)),
            255,
            -1,
        )

    return mascara


def obtener_zonas(forma):
    texto = crear_zona(forma, [
        (0.010, 0.005, 0.285, 0.170),
        (0.010, 0.170, 0.115, 0.400),
        (0.720, 0.005, 0.995, 0.175),
        (0.005, 0.610, 0.095, 0.750),
        (0.005, 0.770, 0.285, 0.998),
        (0.835, 0.735, 0.998, 0.998),
    ])
    brujula = crear_zona(forma, [(0.375, 0.000, 0.625, 0.170)])
    crosshair = crear_zona(forma, [(0.330, 0.260, 0.670, 0.770)])
    inferior = crear_zona(forma, [(0.370, 0.790, 0.625, 0.998)])
    return texto, brujula, crosshair, inferior


def obtener_zonas_circulares(forma):
    return [
        crear_zona(forma, [(0.405, 0.825, 0.500, 0.998)]),
        crear_zona(forma, [(0.495, 0.825, 0.590, 0.998)]),
    ]


def expandir(mascara, radio):
    tamano = 2 * int(radio) + 1
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (tamano, tamano))
    return cv2.dilate(mascara, kernel)


## Reconstrucción del Crosshair

Estima la apertura con las cuatro esquinas y después construye las esquinas, los brazos, la X y las marcas laterales.


In [6]:
def evidencia_crosshair(frame, zona_crosshair):
    fuerte = cv2.bitwise_and(detectar_verde(frame, VERDE_FUERTE), zona_crosshair)
    medio = cv2.bitwise_and(detectar_verde(frame, VERDE_MEDIO), zona_crosshair)
    tenue = cv2.bitwise_and(detectar_verde(frame, VERDE_TENUE), zona_crosshair)

    anclas = cv2.bitwise_or(fuerte, medio)
    tenue_cercano = cv2.bitwise_and(tenue, expandir(anclas, 7))
    evidencia = cv2.bitwise_or(anclas, tenue_cercano)

    horizontal = cv2.morphologyEx(
        evidencia,
        cv2.MORPH_CLOSE,
        cv2.getStructuringElement(cv2.MORPH_RECT, (7, 1)),
    )
    vertical = cv2.morphologyEx(
        evidencia,
        cv2.MORPH_CLOSE,
        cv2.getStructuringElement(cv2.MORPH_RECT, (1, 7)),
    )
    return cv2.bitwise_or(horizontal, vertical)


def plantilla_esquinas(forma, dx, dy, grosor):
    alto, ancho = forma
    cx, cy = ancho // 2, alto // 2
    mascara = np.zeros(forma, dtype=np.uint8)
    largo_h = max(20, int(ancho * 0.030))
    largo_v = max(20, int(alto * 0.045))

    for sx, sy in [(-1, -1), (1, -1), (-1, 1), (1, 1)]:
        x, y = cx + sx * dx, cy + sy * dy
        cv2.line(mascara, (x, y), (x - sx * largo_h, y), 255, grosor, cv2.LINE_AA)
        cv2.line(mascara, (x, y), (x, y - sy * largo_v), 255, grosor, cv2.LINE_AA)

    return mascara


def puntuar_apertura(evidencia, dx, dy):
    alto, ancho = evidencia.shape
    cx, cy = ancho // 2, alto // 2
    soporte = expandir(evidencia, 3) > 0
    largo_h = max(20, int(ancho * 0.030))
    largo_v = max(20, int(alto * 0.045))
    apoyos = []

    for sx, sy in [(-1, -1), (1, -1), (-1, 1), (1, 1)]:
        x, y = cx + sx * dx, cy + sy * dy
        horizontal = np.zeros_like(evidencia)
        vertical = np.zeros_like(evidencia)
        cv2.line(horizontal, (x, y), (x - sx * largo_h, y), 255, 5, cv2.LINE_AA)
        cv2.line(vertical, (x, y), (x, y - sy * largo_v), 255, 5, cv2.LINE_AA)

        ph = horizontal > 0
        pv = vertical > 0
        apoyo_h = np.count_nonzero(ph & soporte) / max(1, np.count_nonzero(ph))
        apoyo_v = np.count_nonzero(pv & soporte) / max(1, np.count_nonzero(pv))
        apoyos.append(min(apoyo_h, apoyo_v))

    buenas = sum(valor >= 0.14 for valor in apoyos)
    fuertes = sum(valor >= 0.30 for valor in apoyos)
    puntaje = np.median(apoyos) * 240 + np.min(apoyos) * 40 + buenas * 35 + fuertes * 20

    if buenas < 2:
        puntaje -= 150

    return float(puntaje), buenas


def estimar_apertura_crosshair(evidencia, anterior=None):
    alto, ancho = evidencia.shape
    if anterior is None:
        anterior = (int(ancho * DX_INICIAL), int(alto * DY_INICIAL))

    mejor = anterior
    mejor_puntaje, mejores_esquinas = puntuar_apertura(evidencia, *anterior)

    for dy in range(int(alto * DY_MINIMO), int(alto * DY_MAXIMO) + 1, PASO_BUSQUEDA):
        for dx in range(int(ancho * DX_MINIMO), int(ancho * DX_MAXIMO) + 1, PASO_BUSQUEDA):
            puntaje, buenas = puntuar_apertura(evidencia, dx, dy)
            puntaje -= 0.035 * (abs(dx - anterior[0]) + abs(dy - anterior[1]))

            if puntaje > mejor_puntaje:
                mejor = (dx, dy)
                mejor_puntaje = puntaje
                mejores_esquinas = buenas

    if mejores_esquinas < 2:
        return anterior, mejores_esquinas, mejor_puntaje

    nueva_x = round(0.60 * anterior[0] + 0.40 * mejor[0])
    nueva_y = round(0.60 * anterior[1] + 0.40 * mejor[1])
    return (int(round(nueva_x / 2) * 2), int(round(nueva_y / 2) * 2)), mejores_esquinas, mejor_puntaje


def dibujar_crosshair_completo(forma, apertura):
    alto, ancho = forma
    cx, cy = ancho // 2, alto // 2
    dx, dy = apertura
    mascara = plantilla_esquinas(forma, dx, dy, GROSOR_CROSSHAIR)

    radio_x = 13
    cv2.line(mascara, (cx-radio_x, cy-radio_x), (cx+radio_x, cy+radio_x), 255, GROSOR_CROSSHAIR, cv2.LINE_AA)
    cv2.line(mascara, (cx-radio_x, cy+radio_x), (cx+radio_x, cy-radio_x), 255, GROSOR_CROSSHAIR, cv2.LINE_AA)
    cv2.circle(mascara, (cx, cy), 5, 255, -1, cv2.LINE_AA)

    brazo_h = max(24, int(dx * 0.78))
    brazo_v = max(16, int(dy * 0.62))
    cv2.line(mascara, (cx-brazo_h, cy), (cx+brazo_h, cy), 255, GROSOR_CROSSHAIR, cv2.LINE_AA)
    cv2.line(mascara, (cx, cy-brazo_v), (cx, cy+brazo_v), 255, GROSOR_CROSSHAIR, cv2.LINE_AA)

    desplazamiento = int(dx * 0.78)
    largo = max(9, int(dy * 0.20))
    for x in (cx-desplazamiento, cx+desplazamiento):
        cv2.line(mascara, (x, cy-largo), (x, cy+largo), 255, GROSOR_CROSSHAIR, cv2.LINE_AA)

    return mascara


def crear_mascara_crosshair(frame, zona_crosshair, anterior=None):
    evidencia = evidencia_crosshair(frame, zona_crosshair)
    apertura, esquinas, puntaje = estimar_apertura_crosshair(evidencia, anterior)
    mascara = dibujar_crosshair_completo(evidencia.shape, apertura)
    mascara = expandir(mascara, MARGEN_CROSSHAIR)
    residuo = cv2.bitwise_and(evidencia, expandir(mascara, RADIO_RESIDUAL_CROSSHAIR))
    mascara = cv2.bitwise_or(mascara, expandir(residuo, 5))
    mascara = cv2.bitwise_and(mascara, zona_crosshair)
    return mascara, apertura, esquinas, puntaje


## Reconstrucción de Círculos

Ajusta centro y radio usando los píxeles de arco y dibuja el contorno completo.


In [7]:
def evaluar_circulo(puntos_x, puntos_y, centro_x, centro_y, radio):
    distancias = np.sqrt((puntos_x-centro_x)**2 + (puntos_y-centro_y)**2)
    error = np.abs(distancias-radio)
    cercanos = error <= 4.0
    return int(np.count_nonzero(cercanos)), float(np.median(error))


def detectar_circulo(frame, zona_circulo, anterior=None):
    medio = detectar_verde(frame, VERDE_MEDIO)
    tenue = detectar_verde(frame, VERDE_TENUE)
    señal = cv2.bitwise_and(cv2.bitwise_or(medio, tenue), zona_circulo)

    ys_zona, xs_zona = np.where(zona_circulo > 0)
    x1, x2 = int(xs_zona.min()), int(xs_zona.max()) + 1
    y1, y2 = int(ys_zona.min()), int(ys_zona.max()) + 1
    region = señal[y1:y2, x1:x2]

    region = cv2.morphologyEx(
        region,
        cv2.MORPH_CLOSE,
        cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5)),
    )
    ys, xs = np.where(region > 0)

    if len(xs) < 12:
        return anterior

    xs_globales = xs.astype(np.float64) + x1
    ys_globales = ys.astype(np.float64) + y1
    alto, ancho = señal.shape
    centro_esperado_x = (x1 + x2) / 2
    centro_esperado_y = (y1 + y2) / 2
    radio_min = int(ancho * 0.018)
    radio_max = int(ancho * 0.050)

    mejor = None
    mejor_puntaje = -1e9
    rango_x = range(int(centro_esperado_x - 28), int(centro_esperado_x + 29), 2)
    rango_y = range(int(centro_esperado_y - 38), int(centro_esperado_y + 39), 2)

    for cy in rango_y:
        for cx in rango_x:
            distancias = np.sqrt((xs_globales-cx)**2 + (ys_globales-cy)**2)
            radio = int(round(np.median(distancias)))

            if radio < radio_min or radio > radio_max:
                continue

            coincidencias, error = evaluar_circulo(xs_globales, ys_globales, cx, cy, radio)
            puntaje = coincidencias * 2.0 - error - 0.04 * abs(cx-centro_esperado_x) - 0.03 * abs(cy-centro_esperado_y)

            if anterior is not None:
                puntaje -= 0.04 * (abs(cx-anterior[0]) + abs(cy-anterior[1]) + abs(radio-anterior[2]))

            if puntaje > mejor_puntaje:
                mejor = (cx, cy, radio)
                mejor_puntaje = puntaje

    if mejor is None:
        return anterior

    if anterior is None:
        return mejor

    return tuple(int(round(0.80*a + 0.20*n)) for a, n in zip(anterior, mejor))


def reconstruir_circulo(frame, zona_circulo, parametros):
    mascara = np.zeros(zona_circulo.shape, dtype=np.uint8)
    if parametros is None:
        return mascara

    centro_x, centro_y, radio = parametros
    cv2.circle(
        mascara,
        (centro_x, centro_y),
        radio + MARGEN_CIRCULO,
        255,
        GROSOR_CIRCULO,
        cv2.LINE_AA,
    )

    señal = cv2.bitwise_and(
        cv2.bitwise_or(detectar_verde(frame, VERDE_MEDIO), detectar_verde(frame, VERDE_TENUE)),
        expandir(zona_circulo, 15),
    )
    conectado = cv2.bitwise_and(señal, expandir(mascara, RADIO_RESIDUAL_CIRCULO))
    mascara = cv2.bitwise_or(mascara, expandir(conectado, 5))
    return mascara


## Máscaras del Bloque

Crea únicamente las máscaras, vistas y diagnósticos del bloque seleccionado.


In [8]:
def crear_atlas_texto(frames):
    cantidad = min(MUESTRAS_ATLAS, len(frames))
    indices = np.linspace(0, len(frames)-1, cantidad).astype(int)
    acumulador = None

    for indice in indices:
        frame = cv2.imread(str(frames[indice]))
        fuerte = detectar_verde(frame, VERDE_FUERTE)
        zona_texto, _, _, _ = obtener_zonas(fuerte.shape)
        muestra = cv2.bitwise_and(fuerte, zona_texto)

        if acumulador is None:
            acumulador = np.zeros(muestra.shape, dtype=np.float32)
        acumulador += (muestra > 0).astype(np.float32)

    return ((acumulador / cantidad) >= UMBRAL_ATLAS).astype(np.uint8) * 255


def preparar_y_enmascarar_bloque(base, numero_bloque=BLOQUE_A_PROBAR):
    base = Path(base)
    todos_los_frames = sorted((base / "frames").glob("*.png"))
    bloques = calcular_bloques(len(todos_los_frames))

    if numero_bloque < 1 or numero_bloque > len(bloques):
        raise ValueError(f"El bloque debe estar entre 1 y {len(bloques)}")

    datos = bloques[numero_bloque - 1]
    carpeta = base / f"bloque_{numero_bloque:02d}"

    if carpeta.exists() and LIMPIAR_BLOQUE_ANTERIOR:
        shutil.rmtree(carpeta)

    for nombre in ("frames", "masks", "vistas", "diagnosticos", "no_hud"):
        (carpeta / nombre).mkdir(parents=True, exist_ok=True)

    frames_bloque = todos_los_frames[datos["inicio_contexto"]:datos["fin_contexto"]]
    atlas = crear_atlas_texto(todos_los_frames)
    guardar_png(carpeta / "atlas_texto.png", atlas)

    apertura = None
    circulos = [None, None]

    for indice, ruta in enumerate(frames_bloque):
        frame = cv2.imread(str(ruta))
        if frame is None:
            raise RuntimeError(f"No se pudo leer {ruta}")

        fuerte = detectar_verde(frame, VERDE_FUERTE)
        medio = detectar_verde(frame, VERDE_MEDIO)
        tenue = detectar_verde(frame, VERDE_TENUE)
        zona_texto, zona_brujula, zona_crosshair, zona_inferior = obtener_zonas(fuerte.shape)

        soporte_texto = cv2.bitwise_and(expandir(atlas, RADIO_ATLAS), zona_texto)
        mascara_texto = cv2.bitwise_and(cv2.bitwise_or(fuerte, medio), soporte_texto)
        mascara_texto = expandir(mascara_texto, 3)

        mascara_crosshair, apertura, esquinas, puntaje = crear_mascara_crosshair(
            frame,
            zona_crosshair,
            apertura,
        )

        mascara_brujula = cv2.bitwise_and(cv2.bitwise_or(fuerte, medio), zona_brujula)
        mascara_brujula = cv2.morphologyEx(
            mascara_brujula,
            cv2.MORPH_CLOSE,
            cv2.getStructuringElement(cv2.MORPH_RECT, (3, 13)),
        )
        mascara_brujula = expandir(mascara_brujula, 5)

        mascara_inferior = cv2.bitwise_and(cv2.bitwise_or(fuerte, medio), zona_inferior)
        mascara_inferior = expandir(mascara_inferior, 4)

        zonas_circulares = obtener_zonas_circulares(fuerte.shape)
        mascaras_circulares = []

        for i, zona_circulo in enumerate(zonas_circulares):
            circulos[i] = detectar_circulo(frame, zona_circulo, circulos[i])
            mascaras_circulares.append(reconstruir_circulo(frame, zona_circulo, circulos[i]))

        mascara = cv2.bitwise_or(mascara_texto, mascara_crosshair)
        mascara = cv2.bitwise_or(mascara, mascara_brujula)
        mascara = cv2.bitwise_or(mascara, mascara_inferior)

        for mascara_circular in mascaras_circulares:
            mascara = cv2.bitwise_or(mascara, mascara_circular)

        zona_hud = cv2.bitwise_or(zona_texto, zona_brujula)
        zona_hud = cv2.bitwise_or(zona_hud, zona_crosshair)
        zona_hud = cv2.bitwise_or(zona_hud, zona_inferior)
        residuo = cv2.bitwise_and(tenue, zona_hud)
        residuo = cv2.bitwise_and(residuo, expandir(mascara, 12))
        mascara = cv2.bitwise_or(mascara, expandir(residuo, 4))
        mascara = (mascara > 0).astype(np.uint8) * 255

        shutil.copy2(ruta, carpeta / "frames" / ruta.name)
        guardar_png(carpeta / "masks" / ruta.name, mascara)

        rojo = np.zeros_like(frame)
        rojo[:, :, 2] = 255
        mezcla = cv2.addWeighted(frame, 0.60, rojo, 0.40, 0)
        vista = frame.copy()
        vista[mascara > 0] = mezcla[mascara > 0]
        guardar_png(carpeta / "vistas" / ruta.name, vista)

        if GUARDAR_DIAGNOSTICOS:
            diagnostico = frame.copy()
            plantilla = dibujar_crosshair_completo(mascara.shape, apertura)
            diagnostico[plantilla > 0] = (255, 0, 255)

            for parametros in circulos:
                if parametros is not None:
                    cx, cy, radio = parametros
                    cv2.circle(diagnostico, (cx, cy), radio, (0, 255, 255), 3, cv2.LINE_AA)
                    cv2.circle(diagnostico, (cx, cy), 4, (255, 255, 0), -1, cv2.LINE_AA)

            cv2.putText(
                diagnostico,
                f"dx={apertura[0]} dy={apertura[1]} esquinas={esquinas} puntaje={puntaje:.1f}",
                (20, 30),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.65,
                (255, 255, 0),
                2,
                cv2.LINE_AA,
            )
            guardar_png(carpeta / "diagnosticos" / ruta.name, diagnostico)

        if (indice + 1) % 50 == 0:
            print(f"Bloque {numero_bloque:02d}: {indice + 1}/{len(frames_bloque)}")

    nombres_principales = [
        todos_los_frames[i].name
        for i in range(datos["inicio"], datos["fin"])
    ]
    (carpeta / "frames_finales.txt").write_text(
        "\n".join(nombres_principales) + "\n",
        encoding="utf-8",
    )

    print(f"Bloque preparado: {carpeta}")
    print(f"Frames procesados: {len(frames_bloque)}")
    return carpeta


## Probar el Bloque 1

Extrae los frames si hace falta, muestra el plan y crea solamente el bloque 1.


In [9]:
def probar_bloque_1():
    base = extraer_frames()
    mostrar_bloques(base)
    carpeta = preparar_y_enmascarar_bloque(base, numero_bloque=1)
    print("Revisa las carpetas vistas y diagnosticos del bloque_01")
    return carpeta


carpeta_prueba = probar_bloque_1()


Se usarán 3300 frames existentes
Frames disponibles: 3300
Bloques necesarios: 7
Bloque 01: 500 principales, 510 para procesar
Bloque 02: 500 principales, 520 para procesar
Bloque 03: 500 principales, 520 para procesar
Bloque 04: 500 principales, 520 para procesar
Bloque 05: 500 principales, 520 para procesar
Bloque 06: 500 principales, 520 para procesar
Bloque 07: 300 principales, 310 para procesar


KeyboardInterrupt: 